### RAG Pipeline: Data Ingestion to Vector DB Pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [25]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./data")

Found 4 PDF files to process

Processing: dog_diseases.pdf
  ✓ Loaded 7 pages

Processing: hb_prince.pdf
  ✓ Loaded 1 pages

Processing: pl_stone.pdf
  ✓ Loaded 1 pages

Processing: web react.pdf
  ✓ Loaded 12 pages

Total documents loaded: 21


In [26]:
#see al documents with your added metadata
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-08-25T10:29:16+05:30', 'author': 'Krunal Solanki, Abhi Desai, Milind Dalvi and Harsh Jani', 'keywords': 'Canine, dog, diseases, rabies, leptospirosis, kennel cough', 'moddate': '2023-08-25T10:29:30+05:30', 'subject': 'Review on important diseases of Dogs: At glance', 'title': 'Review on important diseases of Dogs: At glance', 'source': 'data\\pdf\\dog_diseases.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'dog_diseases.pdf', 'file_type': 'pdf'}, page_content='~ 1 ~ \nInternational Journal of Veterinary Sciences and Animal Husbandry 2023; 8(2): 01-07 \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nISSN: 2456-2912 \nVET 2023; 8(2): 01-07 \n© 2023 VET \nwww.veterinarypaper.com \nReceived: 03-01-2023 \nAccepted: 04-02-2023 \n \nKrunal Solanki \nResearch Scientist, Veterinary \nPathologist, Ribosome Research \nCentre Pvt Ltd., Kim, Gujarat, \nIndia \n \n

#### Chunking usng textsplitter

In [27]:
# this will split the documents into smaller chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f" Split {len(documents)} into {len(split_docs)} chunks ")

    #Show example of a chunk

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]} ...")
        print(f"Metadata : {split_docs[0].metadata}")
        
    return split_docs


In [23]:
chunks=split_documents(all_pdf_documents)
chunks

 Split 21 into 84 chunks 

Example chunk:
Content:~ 1 ~ 
International Journal of Veterinary Sciences and Animal Husbandry 2023; 8(2): 01-07 
 
  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
ISSN: 2456-2912 
VET 2023; 8(2): 01-07 
© 2023 VET 
www.veterinarypaper.com ...
Metadata : {'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-08-25T10:29:16+05:30', 'author': 'Krunal Solanki, Abhi Desai, Milind Dalvi and Harsh Jani', 'keywords': 'Canine, dog, diseases, rabies, leptospirosis, kennel cough', 'moddate': '2023-08-25T10:29:30+05:30', 'subject': 'Review on important diseases of Dogs: At glance', 'title': 'Review on important diseases of Dogs: At glance', 'source': 'data\\pdf\\dog_diseases.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'dog_diseases.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-08-25T10:29:16+05:30', 'author': 'Krunal Solanki, Abhi Desai, Milind Dalvi and Harsh Jani', 'keywords': 'Canine, dog, diseases, rabies, leptospirosis, kennel cough', 'moddate': '2023-08-25T10:29:30+05:30', 'subject': 'Review on important diseases of Dogs: At glance', 'title': 'Review on important diseases of Dogs: At glance', 'source': 'data\\pdf\\dog_diseases.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'dog_diseases.pdf', 'file_type': 'pdf'}, page_content='~ 1 ~ \nInternational Journal of Veterinary Sciences and Animal Husbandry 2023; 8(2): 01-07 \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nISSN: 2456-2912 \nVET 2023; 8(2): 01-07 \n© 2023 VET \nwww.veterinarypaper.com \nReceived: 03-01-2023 \nAccepted: 04-02-2023 \n \nKrunal Solanki \nResearch Scientist, Veterinary \nPathologist, Ribosome Research \nCentre Pvt Ltd., Kim, Gujarat, \nIndia \n \n

#### Embedding and Vectore StoreDB

In [ ]:
#for embedding we will use setence transformer an opensource an open source model in hugging face

# cmd run :  uv  add -r requirments.txt

In [28]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # every entry into the Vdb will need an id
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity # we'll use cosine_similarity for retrival and match  

In [ ]:
# Modular  coding method (Classes and Objects)


In [29]:
class EmbeddingManager:

    def __init__(self,model_name:str="all-MiniLM-L6-v2"): # hugging face model , to create embeddings (has roughly 385 dimesiions)
        
        """ initialise the embedding manager"""

        self.model_name =model_name
        self.model= None
        self._load_model()#imp (its going to load the model)

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")#(default is 384 dimesions)
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


d:\gstudy\Personal projects\AI ML projects\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GHYAN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 574.62it/s, Materiali

Model loaded successfully. Embedding dimension: 384


### VectorStore

In [30]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [32]:
#check chunks again
chunks

[Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2023-08-25T10:29:16+05:30', 'author': 'Krunal Solanki, Abhi Desai, Milind Dalvi and Harsh Jani', 'keywords': 'Canine, dog, diseases, rabies, leptospirosis, kennel cough', 'moddate': '2023-08-25T10:29:30+05:30', 'subject': 'Review on important diseases of Dogs: At glance', 'title': 'Review on important diseases of Dogs: At glance', 'source': 'data\\pdf\\dog_diseases.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'dog_diseases.pdf', 'file_type': 'pdf'}, page_content='~ 1 ~ \nInternational Journal of Veterinary Sciences and Animal Husbandry 2023; 8(2): 01-07 \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nISSN: 2456-2912 \nVET 2023; 8(2): 01-07 \n© 2023 VET \nwww.veterinarypaper.com \nReceived: 03-01-2023 \nAccepted: 04-02-2023 \n \nKrunal Solanki \nResearch Scientist, Veterinary \nPathologist, Ribosome Research \nCentre Pvt Ltd., Kim, Gujarat, \nIndia \n \n

In [33]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 84 texts...


Batches: 100%|██████████| 3/3 [00:05<00:00,  1.74s/it]


Generated embeddings with shape: (84, 384)
Adding 84 documents to vector store...
Successfully added 84 documents to vector store
Total documents in collection: 84


### Retriever Pipeline From VectorStore

In [34]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [35]:

rag_retriever

In [37]:
rag_retriever.retrieve("Who is harry potter")

Retrieving documents for query: 'Who is harry potter'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_2507aaee_59',
  'content': 'Harry Potter and the Philosopher’s Stone \nHarry Potter and the Philosopher’s Stone introduces readers to the magical world and follows the \nearly life of Harry Potter, an orphan living with his cruel relatives, the Dursleys. Harry is treated \npoorly and kept unaware of his true identity until his eleventh birthday, when he discovers that \nhe is a wizard. He learns that his parents were powerful wizards who were killed by the dark \nwizard Lord Voldemort, and that Harry himself survived the attack, earning a lightning-shaped \nscar on his forehead. \nHarry is invited to attend Hogwarts School of Witchcraft and Wizardry, where he enters a world \nfilled with magic, mystery, and wonder. On his journey to Hogwarts, he befriends Ron Weasley \nand later Hermione Granger, who become his closest companions. At Hogwarts, Harry is sorted \ninto Gryffindor House and quickly gains attention for his natural talent in Quidditch, becoming \nthe youngest Se

In [38]:
rag_retriever.retrieve("What diseases can my dog have")

Retrieving documents for query: 'What diseases can my dog have'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.51it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_31694eb6_3',
  'content': 'and dogs has been shown to be good for their physical and mental health, there is a lack of \nknowledge among owners about the various diseases that can infect dogs. Dogs and humans \nshare a close household environment, making them potential susceptible f or various diseases. \nThe virus can spread to people through their nails, feces, urine, saliva, and other bodily fluids. \nIt is crucial that dog owners learn about canine disease and zoonoses, their potential \ntransmission routes, and preventative measures. M any bacterial, viral, fungal, and parasitic \nillnesses as well as parasite infestations frequently spread from sick pets to humans. Some of \nthese illnesses are specific to the species or to closely related species, while others may be \nzoonotic diseases that should be taken seriously but are frequently overlooked. So, the focus of \nthis analysis is on the canine illnesses that are significant in India. The four main categories of',

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [46]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [48]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [49]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [50]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Who is harry potter")

Retrieving documents for query: 'Who is harry potter'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.67it/s]

Generated embeddings with shape: (1, 384)


Retrieved 4 documents (after filtering)


[{'id': 'doc_2507aaee_59',
  'content': 'Harry Potter and the Philosopher’s Stone \nHarry Potter and the Philosopher’s Stone introduces readers to the magical world and follows the \nearly life of Harry Potter, an orphan living with his cruel relatives, the Dursleys. Harry is treated \npoorly and kept unaware of his true identity until his eleventh birthday, when he discovers that \nhe is a wizard. He learns that his parents were powerful wizards who were killed by the dark \nwizard Lord Voldemort, and that Harry himself survived the attack, earning a lightning-shaped \nscar on his forehead. \nHarry is invited to attend Hogwarts School of Witchcraft and Wizardry, where he enters a world \nfilled with magic, mystery, and wonder. On his journey to Hogwarts, he befriends Ron Weasley \nand later Hermione Granger, who become his closest companions. At Hogwarts, Harry is sorted \ninto Gryffindor House and quickly gains attention for his natural talent in Quidditch, becoming \nthe youngest Se

### Integration Vectordb Context pipeline With LLM output

In [53]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [54]:
answer=rag_simple("Who is harry potter?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Who is harry potter?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Harry Potter is an orphan who is a powerful wizard, the son of two powerful wizards who were killed by the dark wizard Lord Voldemort.


In [55]:
answer=rag_simple("Tell me about some dog diseases?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Tell me about some dog diseases?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


According to the provided context, some significant dog diseases in India include:

1. Rabies
2. Canine Parvoviral gastroenteritis
3. Canine Distemper
4. Canine Coronaviral infection
5. Canine Rotavirus infection
6. Canine Herpesvirus infection
7. Canine Leptospirosis
8. Canine Brucellosis
9. Transmissible venereal tumors (TVT)
10. Kennel cough
11. Pyoderma


### Enhanced RAG Pipeline Features

In [57]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Worst Dog diseases", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Worst Dog diseases'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: According to the given context, some of the worst dog diseases include:

1. Rabies
2. Canine Parvoviral gastroenteritis
3. Canine Distemper
4. Canine Leptospirosis
5. Canine Brucellosis
6. Transmissible venereal tumors (TVT)
7. Kennel cough
8. Canine Coronaviral infection
9. Canine Rotavirus infection
10. Canine Herpesvirus infection
Sources: [{'source': 'dog_diseases.pdf', 'page': 0, 'score': 0.3424670100212097, 'preview': 'and dogs has been shown to be good for their physical and mental health, there is a lack of \nknowledge among owners about the various diseases that can infect dogs. Dogs and humans \nshare a close household environment, making them potential susceptible f or various diseases. \nThe virus can spread to...'}, {'source': 'dog_diseases.pdf', 'page': 0, 'score': 0.21877682209014893, 'preview': 'like, Rabies, Canine Parvoviral gastroenteritis, Canine Distemper, Canine Coronaviral infection, Canine \nRotavirus infection, Canine Herpesvirus infection, Canine Lepto

In [61]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Tell me about Canine Coronaviral infection", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Tell me about Canine Coronaviral infection'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
~ 3 ~ 
International Journal of Veterinary Sciences and Animal Husbandry https://www.veterinarypaper.com 
belongs to the family Coronaviridae. When view ed from 
above using an electron microscope, the virus has a ring of 
projections that resemble a 

coronet or a miniature crown 
composed of ornaments attached to a metal ring. There are 
numerous coronavirus kinds, each of which affects distinct 
animal species , including humans. Coronavirus transmission 
is caused by overcrowding and poor sanitation. One to four 
days pass between consumption and clinical manifestations. 
In most canines, the length of illness ranges from two to ten 
days. Secondary infections caus ed by bacteria, parasites, and 
other viruses can cause disease and recovery to be prolonged. 
Dogs may be disease carriers for up to six months (180 days) 
following infection. The majority of canine coronavirus 
infections are subclinical and manifest little  clinical

enhances the host's susceptibility to secondary infections, the 
leading cause of death (Joshi et al., 2022a; Joshi et al., 2022b) 
[22, 23 ]. CDV is most lethal in young puppies, causing rapid 
death a few days after infection. 
 
Canine Coronaviral infection 
CCoV, or canine coronavirus illness,  is 